In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/dgomonov/new-york-city-airbnb-open-data/AB_NYC_2019.csv
/kaggle/input/datasets/dgomonov/new-york-city-airbnb-open-data/New_York_City_.png


In [2]:
airbnb=pd.read_csv("/kaggle/input/datasets/dgomonov/new-york-city-airbnb-open-data/AB_NYC_2019.csv")
#sample rows
airbnb.head()


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [3]:
#print the null values
print(airbnb.isnull().sum())

id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64


In [4]:
#get the summary 
# airbnb.describe()
print(airbnb.describe())

                 id       host_id      latitude     longitude         price  \
count  4.889500e+04  4.889500e+04  48895.000000  48895.000000  48895.000000   
mean   1.901714e+07  6.762001e+07     40.728949    -73.952170    152.720687   
std    1.098311e+07  7.861097e+07      0.054530      0.046157    240.154170   
min    2.539000e+03  2.438000e+03     40.499790    -74.244420      0.000000   
25%    9.471945e+06  7.822033e+06     40.690100    -73.983070     69.000000   
50%    1.967728e+07  3.079382e+07     40.723070    -73.955680    106.000000   
75%    2.915218e+07  1.074344e+08     40.763115    -73.936275    175.000000   
max    3.648724e+07  2.743213e+08     40.913060    -73.712990  10000.000000   

       minimum_nights  number_of_reviews  reviews_per_month  \
count    48895.000000       48895.000000       38843.000000   
mean         7.029962          23.274466           1.373221   
std         20.510550          44.550582           1.680442   
min          1.000000           0.00

In [5]:
#  use the filtered dataframe
data = airbnb[airbnb['price'] > 0].copy()

# cap price and minimum_nights at their 99th percentile
price_cap = data['price'].quantile(0.99)
nights_cap = data['minimum_nights'].quantile(0.99)

# create two columns additional for the data 
data['price_capped'] = data['price'].clip(upper=price_cap)
data['minimum_nights_capped'] = data['minimum_nights'].clip(upper=nights_cap)

# sanity check — did it work?
print(data['price_capped'].max())
print(data['minimum_nights_capped'].max())

799
45


In [6]:
# fill reviews_per_month nulls with 0  no reviews means literally zero reviews per month
data['reviews_per_month'] = data['reviews_per_month'].fillna(0)

# sanity check
print(data['reviews_per_month'].isnull().sum())

0


In [7]:
# days we had reveiws 
data['has_reviews'] = data['number_of_reviews'] > 0
print(data['has_reviews'].value_counts())

has_reviews
True     38833
False    10051
Name: count, dtype: int64


In [8]:
# convert availability_365 (days) into months available
data['months_available'] = data['availability_365'] / 30
data['demand_density'] = data['reviews_per_month'] / data['months_available'].replace(0, np.nan)

# summary of the demand desnsity figure 
print(data['demand_density'].describe())

count    31354.000000
mean         1.668995
std          7.861610
min          0.000000
25%          0.027124
50%          0.205263
75%          0.744000
max        305.100000
Name: demand_density, dtype: float64


In [9]:
# check rows with the most extreme demand_density
print(data.nlargest(10, 'demand_density')[['availability_365', 'reviews_per_month', 'demand_density']])

       availability_365  reviews_per_month  demand_density
43790                 1              10.17           305.1
37912                 1               6.95           208.5
35526                 1               6.84           205.2
43960                 1               6.71           201.3
35377                 1               6.50           195.0
36042                 1               5.82           174.6
21704                 1               5.70           171.0
43179                 1               5.62           168.6
45753                 1               5.38           161.4
40686                 1               5.37           161.1


In [10]:
# only trust demand_density when there's enough availability to make the ratio meaningful
data['demand_density'] = np.where(
    data['availability_365'] >= 30,
    data['reviews_per_month'] / (data['availability_365'] / 30),
    np.nan
)

print(data['demand_density'].describe())
print(data.nlargest(10, 'demand_density')[['availability_365', 'reviews_per_month', 'demand_density']])

count    26189.000000
mean         0.392272
std          0.688511
min          0.000000
25%          0.020625
50%          0.148052
75%          0.447368
max         10.041667
Name: demand_density, dtype: float64
       availability_365  reviews_per_month  demand_density
33943                36              12.05       10.041667
40823                30               9.64        9.640000
37093                41              13.11        9.592683
46937                45              14.00        9.333333
35707                31               9.63        9.319355
17771                31               9.30        9.000000
40316                34               9.13        8.055882
43791                42              11.25        8.035714
23691                30               7.84        7.840000
41194                30               7.79        7.790000


# NYC Airbnb Open Data — Data Wrangling & Feature Engineering

**Dataset:** [New York City Airbnb Open Data (2019)](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data)
**Rows / Columns:** 48,895 rows × 16 columns
**Goal:** Clean the raw listing data and engineer features that capture real signal (pricing behavior, host activity, listing demand) before any modeling.


## 1. Initial Inspection

Before summary statistics, checked shape, dtypes, and nulls first — `.describe()` alone hides missingness and only covers numeric columns.

```python
import pandas as pd
import numpy as np

airbnb = pd.read_csv('AB_NYC_2019.csv')
print(airbnb.shape)          # (48895, 16)
print(airbnb.dtypes)
print(airbnb.isnull().sum())
```

**Null counts found:**

| Column | Nulls | Why |
|---|---|---|
| `name` | 16 | Minor, likely blank listing titles |
| `host_name` | 21 | Minor, blank field |
| `last_review` | 10,052 | Structural — listing has never been reviewed |
| `reviews_per_month` | 10,052 | Same structural cause as above |

Note that `last_review` and `reviews_per_month` share the exact same null count — not a coincidence. Both are null for the same reason: a listing with **zero reviews ever** has no last review date and no review rate to compute. This is *structural* missingness, not random, which changes how it should be handled (see §3).



## 2. Outlier Diagnosis

`price` showed **zero nulls**, but that doesn't mean it's clean — `.describe()` revealed the real issue:

```
price:            min = 0        | 75th pct = 175    | max = 10,000
minimum_nights:   75th pct = 5   | max = 1,250 (≈3.4 years)
calculated_host_listings_count: 75th pct = 2 | max = 327
```

**Diagnosis:**
- `price = 0` → almost certainly inactive/blocked listings, not real prices (data was scraped by Inside Airbnb, not self-reported for billing).
- `minimum_nights = 1250` → likely a host effectively disabling short-term bookings without deleting the listing (switched to long-term rental).
- `calculated_host_listings_count = 327` → a property management company, not an individual host.

**Fix applied — winsorizing (percentile capping) rather than deleting:**

```python
data = airbnb[airbnb['price'] > 0].copy()   # drop invalid zero-price rows

price_cap = data['price'].quantile(0.99)          # 799.0
nights_cap = data['minimum_nights'].quantile(0.99) # 45.0

data['price_capped'] = data['price'].clip(upper=price_cap)
data['minimum_nights_capped'] = data['minimum_nights'].clip(upper=nights_cap)
```

Capped values are stored in **new columns**, keeping the original `price` and `minimum_nights` intact — always preserve the raw data so any transformation can be audited or reversed.

---

## 3. Structural vs. True Missingness

Key distinction applied throughout this project:

- **Structural null** → the "unknown" has a knowable, logical value. Example: `reviews_per_month` is null because there are 0 reviews — the correct fill is `0`, not the column mean.
- **True/unknowable null** → there's no logical value to fill in. Example: `last_review` has no meaningful date to impute for a never-reviewed listing — left as null (or later converted to a boolean flag).

```python
data['reviews_per_month'] = data['reviews_per_month'].fillna(0)
data['has_reviews'] = data['number_of_reviews'] > 0
```

---

## 4. Feature Engineering: `demand_density`

**Goal:** capture how much a listing is actually used *relative to* how available it is — a listing open 365 days/year with barely any reviews behaves very differently from one open only 30 days/year that's constantly booked.

**First attempt** (naive ratio):

```python
data['months_available'] = data['availability_365'] / 30
data['demand_density'] = data['reviews_per_month'] / data['months_available'].replace(0, np.nan)
```

Result: `max = 305.1` — a clear outlier problem. Diagnosed the cause directly rather than blindly capping:

```python
data.nlargest(10, 'demand_density')[['availability_365', 'reviews_per_month', 'demand_density']]
```

Every one of the top 10 rows had `availability_365 = 1`. **Root cause:** dividing by a near-zero denominator inflates the ratio into a meaningless number — this is a formula problem, not just an "extreme value" problem, so winsorizing alone would only mask it.

**Fix — apply a minimum-availability threshold before trusting the ratio:**

```python
data['demand_density'] = np.where(
    data['availability_365'] >= 30,
    data['reviews_per_month'] / (data['availability_365'] / 30),
    np.nan
)
```

Result: `max` dropped from 305.1 → **10.04**, and the top rows now show realistic listings (30–45 days available, high genuine review activity) instead of divide-by-near-zero artifacts.

**Important nuance:** a computed `demand_density = 0` (listing was available but got zero reviews) is a *real signal* and should **not** be converted to `NaN` — only values that are genuinely uncomputable (availability below the threshold) are marked null. Conflating "true zero" with "unknown" would throw away real findings.

---

## 5. Summary of Decisions

| Issue | Decision | Reasoning |
|---|---|---|
| `price = 0` | Drop rows | Small number of rows, clearly invalid/inactive listings |
| `price`, `minimum_nights` outliers | Cap at 99th percentile (new columns) | Preserves row count, limits extreme-value influence on models |
| `reviews_per_month` nulls | Fill with 0 | Structural — no reviews means a true rate of zero |
| `last_review` nulls | Leave null / flag with `has_reviews` | No logical date to impute |
| `demand_density` outliers | Threshold + conditional formula, not capping | Root cause was a broken ratio (near-zero denominator), not just extreme values |



## Next Steps

- [ ] Visualize `price_capped` distribution and geographic patterns (`latitude`/`longitude` by `neighbourhood_group`)
- [ ] One-hot encode `room_type`, `neighbourhood_group`
- [ ] Build a baseline regression model predicting `price_capped`
- [ ] Compare against a tree-based model (Random Forest), consistent with the California Housing project approach